## Denoising Autoencoders

In this example a denoising autoencoder is trained using MNIST with corruption introduced into the input, while the uncorrupted images are used at the output. The autoencoder learns to remove the noise.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm.notebook import trange
import torch.nn.functional as F

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

### Prepare the Data

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# Add noise to the images
def add_noise(images, noise_factor=0.5):
    noisy_images = images + noise_factor * torch.randn(*images.shape)
    noisy_images = torch.clip(noisy_images, 0., 1.)
    return noisy_images

In [ ]:
train_im_noisy = []
train_images = []

for images, _ in train_loader:
    noisy_images = add_noise(images)
    train_im_noisy.append(noisy_images)
    train_images.append(images)

train_im_noisy = torch.cat(train_im_noisy)
train_images = torch.cat(train_images)

test_im_noisy = []
test_images = []

for images, _ in test_loader:
    noisy_images = add_noise(images)
    test_im_noisy.append(noisy_images)
    test_images.append(images)

test_im_noisy = torch.cat(test_im_noisy)
test_images = torch.cat(test_images)

In [ ]:
train_images[0]

In [ ]:
train_dataset = TensorDataset(train_im_noisy.to(device), train_images.to(device))
test_dataset = TensorDataset(test_im_noisy.to(device), test_images.to(device))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

Add the random noise to the training data

### Build the Autoencoder

In this example, convolutional layers will be used and the output later will be changed to used sigmoid activation rather than linear. Given all the numerical pixel intensities lie between 0 and 1 it makes sense to mirror this in the choice of output activation function. The choice of sigmoid output means that binary crossentropy is the optimal choice of loss function.

In [ ]:
# Define the Denoising Autoencoder (DAE)
class DAE(nn.Module):
    def __init__(self):
        super(DAE, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0),  # 28x28 -> 14x14
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0)  # 14x14 -> 7x7
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 7x7 -> 7x7
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 7x7 -> 14x14
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 14x14 -> 28x28
            nn.Conv2d(32, 1, kernel_size=3, padding=1),  # 28x28 -> 28x28
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

### Train the model

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    history = {'train_loss': [], 'val_loss': []}

    for epoch in trange(num_epochs):
        model.train()
        train_loss = 0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss = train_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * images.size(0)

        val_loss = val_loss / len(test_loader.dataset)
        history['val_loss'].append(val_loss)

        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

    return history

In [ ]:
no_epochs = 200

In [ ]:
dae_auto = DAE().to(device)
dae_history = train_model(dae_auto, train_loader, test_loader, no_epochs)

### Display Test Images

In [ ]:
def reconstruct(model, test_loader):
    model.eval()
    reconstructions = []
    with torch.no_grad():
        for images, _ in test_loader:
            outputs = model(images)
            reconstructions.append(outputs.cpu().numpy())

    return np.concatenate(reconstructions, axis=0)

In [ ]:
dae_reconstruction = reconstruct(dae_auto, test_loader)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Display original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(test_loader.dataset[i][1].view(28, 28).cpu().detach().numpy(), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display shallow
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(test_loader.dataset[i][0].view(28, 28).cpu().detach().numpy(), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display deep
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(dae_reconstruction[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.savefig('daereconstruct.png', dpi=300, bbox_inches='tight')